# Stage 2b — How good are the YOLOX detections and RTMPose keypoints, per video?

Reads the Stage 2 output (`outputs/02_pose/<RUN_NAME>/npz/*.npz`) and your Stage 1 labels, and answers:

| Question | Where |
|---|---|
| Is the **patient detected** at the labelled frames? How tight is the box (IoU)? | cell 4, `per_video_summary.csv` |
| How many **people per frame**, how often **nobody / flicker**, how many weak boxes? | cell 4 |
| Would a simple rule already find the patient (**most central / highest score / largest** box)? | cell 4 (preview of Stage 4 rule *a*) |
| Which **joints** does RTMPose see confidently, especially **ankles / heels / toes**? | cell 5 |
| Are the patient's ankle keypoints **stable over time** (jitter) around each labelled frame? | cell 6 |
| Everything about **one video**: timelines, ankle trajectory, overlay images | cell 7 |
| Which videos **need attention**, and why? | cell 8, `flags.csv` |
| **Part B:** which box is the **patient** and which are **observers**, in every frame? | cells 10–13, `identity/` |
| **Part B:** where might the patient label have **switched to an observer**? → review with `review_patient.py` | cells 12–14 |

### Running on your laptop: where to put the files
Put this notebook in a project folder, make a `data/` folder next to it, and put everything **inside `data/`**.
Sub-folder names don't matter: the notebook searches all of `data/`.
```
FOG/                                    ← start Jupyter here (this notebook is in this folder)
├── 02b_analysis.ipynb
├── review_patient.py                   ← Part B: review suspected label switches (run in a terminal)
├── data/
│   ├── videos/                         ← the 73 Figshare videos (only needed for the overlay images)
│   ├── labels/
│   │   ├── patient_labels.csv          ← Stage 1 labels
│   │   └── video_layout.csv            ← optional: columns video_id,layout (e.g. koridor / lab)
│   └── stage2/                         ← the Stage 2 output downloaded from Kaggle, unzipped here
│       └── ... /balanced_s2_det0.1/npz/<video_id>.npz      (any depth is fine)
└── outputs/                            ← created by this notebook
    └── 02b_analysis/balanced_s2_det0.1/   ← all tables and figures
```
**Getting the Stage 2 output from Kaggle.** Either one works; unzip it into `data/stage2/`:
* the run zip `outputs/02_pose_<RUN_NAME>.zip` (open the Stage 2 notebook → **Output** tab → download that file), or
* **Download All** on the Output tab (a zip of the whole `/kaggle/working`).

Only `npz/` is required. You can delete `preview/` and `videos/` from the download to save space.

**Install once** (in a terminal, in the same Python environment Jupyter uses):
```
pip install numpy pandas matplotlib opencv-python scipy jupyter
```
Then run `jupyter notebook` (or open the folder in VS Code), open `02b_analysis.ipynb`, and choose **Run All**.
If your files are somewhere else, set `DATA_DIR` (or the `*_OVERRIDE` paths) in cell 1.

### On Kaggle (alternative)
No GPU needed (Accelerator: None is fine). **Add Input**:
1. **Notebook Output → your Stage 2 notebook** (the version that finished),
2. your **videos dataset** (only for the overlay images; numbers work without it),
3. the dataset with **`patient_labels.csv`**.

Optional: a CSV with columns `video_id,layout` (e.g. `koridor` / `lab`) → results are also grouped by layout (Stage 6).

### Output (one place)
```
<project folder>/outputs/02b_analysis/<RUN_NAME>/     (on Kaggle: /kaggle/working/outputs/02b_analysis/<RUN_NAME>/)
├── per_video_summary.csv     ← one row per video: detection + pose + flags  (main table)
├── labelled_frames.csv       ← one row per labelled frame: matched box, IoU, score, rank, ankle confidence
├── joint_confidence.csv      ← mean confidence per Halpe-26 joint: all people vs patient
├── snippets.csv              ← patient followed ±2 s around each labelled frame: coverage, ankle jitter
├── flags.csv                 ← videos needing attention + reasons
└── figures/*.png             ← every plot shown below, plus figures/videos/<video_id>_*.png
```

## 1. Settings

In [ ]:
DATA_DIR            = "data"  # everything is searched inside this folder (relative to this notebook)
OUTPUT_ROOT         = "."     # results go to <OUTPUT_ROOT>/outputs/02b_analysis/<RUN_NAME>/
RUN_NAME            = None   # Stage 2 run to analyse, e.g. "balanced_s2_det0.1"; None = the one with most videos
POSE_DIR_OVERRIDE   = None   # e.g. r"D:\FOG\data\stage2\balanced_s2_det0.1\npz"
VIDEO_DIR_OVERRIDE  = None   # videos are only needed for overlay images
LABELS_CSV_OVERRIDE = None
META_CSV_OVERRIDE   = None   # optional CSV with video_id + layout (or any grouping column)
DATASET_HINT        = ""     # only use input paths containing this text

STRONG_DET = 0.5    # a box with score >= this counts as "a person" in the people-per-frame statistics
IOU_MATCH  = 0.5    # label box vs detection: IoU >= this = patient detected
KPT_THR    = 0.3    # keypoint counts as "seen" if its score >= this
SNIPPET_S  = 2.0    # follow the patient this many seconds before/after each labelled frame (cell 6)
LINK_IOU   = 0.3    # IoU needed to link the patient box from one processed frame to the next
VIEW_VIDEO = None   # cell 7: video_id to inspect; None = the video with the lowest patient recall

# flag thresholds (cell 8)
FLAG_RECALL     = 0.75   # patient detected in fewer labelled frames than this
FLAG_ANKLE_CONF = 0.5    # mean patient ankle confidence below this
FLAG_NO_PERSON  = 0.10   # more than this share of frames without any strong box
FLAG_COVERAGE   = 0.80   # patient could be followed through less than this share of snippet frames

CELLS_DONE = set()
CELL_TITLES = {"settings": "1. Settings", "paths": "2. Paths", "functions": "3. Functions",
               "detection": "4. Detection", "pose": "5. Pose", "snippets": "6. Snippets",
               "video": "7. One video", "flags": "8. Flags", "save": "9. Save",
               "id-settings": "10. Part B settings", "tracker": "11. Trackers",
               "identity": "12. Patient vs observer", "id-video": "13. One video: who is who",
               "review": "14. Review results"}
def _done(name):
    CELLS_DONE.add(name)
def _require(*names):
    missing = [CELL_TITLES[n] for n in names if n not in CELLS_DONE]
    if missing:
        raise RuntimeError("Not finished yet: " + ", ".join(missing) + ". Run those cells in order and fix the "
                           "FIRST error you see.")
_done('settings')

## 2. Find the Stage 2 results, labels and videos

In [ ]:
if "CELLS_DONE" not in globals():
    raise RuntimeError("Run cell 1 (Settings) first: the kernel was restarted or cells were skipped.")
_require('settings')

import csv, json, os, re, shutil, sys, time
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from IPython.display import display

IN_KAGGLE  = Path("/kaggle/working").exists() or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
INPUT_ROOT = Path("/kaggle/input") if IN_KAGGLE else Path(os.environ.get("FOG_INPUT_ROOT", DATA_DIR)).resolve()
WORK_ROOT  = Path("/kaggle/working") if IN_KAGGLE else Path(os.environ.get("FOG_WORK_ROOT", OUTPUT_ROOT)).resolve()
LAYOUT_HELP = f"""
Expected layout (everything inside {INPUT_ROOT}):
  {INPUT_ROOT.name}/videos/...                         the original videos (only for images)
  {INPUT_ROOT.name}/labels/patient_labels.csv          Stage 1 labels
  {INPUT_ROOT.name}/stage2/.../<RUN_NAME>/npz/*.npz    Stage 2 output from Kaggle, unzipped
Notebook is running in: {Path.cwd()}"""
if not IN_KAGGLE and not INPUT_ROOT.exists():
    raise RuntimeError(f"Data folder {INPUT_ROOT} does not exist. Create it next to this notebook "
                       f"(or set DATA_DIR in cell 1).{LAYOUT_HELP}")
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".m4v", ".mpg", ".mpeg", ".wmv"}

def npz_dirs():
    if POSE_DIR_OVERRIDE:
        return [Path(POSE_DIR_OVERRIDE)]
    roots = [INPUT_ROOT, WORK_ROOT / "outputs" / "02_pose"]   # never scan the whole laptop
    found = set()
    for root in roots:
        if root.exists():
            for f in root.rglob("*.npz"):
                if f.parent.name == "npz" and DATASET_HINT in f.as_posix():
                    found.add(f.parent.resolve())
    return sorted(found)

cands = [(p, len(list(p.glob("*.npz")))) for p in npz_dirs()]
STAGE2_RUN_DIRS = {p.parent for p, _ in cands}
def in_stage2(p):
    """True for files inside a Stage 2 run folder (previews, rendered videos, manifest...)."""
    p = Path(p).resolve()
    return "outputs/02" in p.as_posix() or any(d in p.parents for d in STAGE2_RUN_DIRS)
cands = [c for c in cands if c[1] > 0]
if RUN_NAME:
    cands = [c for c in cands if c[0].parent.name == RUN_NAME]
if not cands:
    raise RuntimeError("No Stage 2 results (.npz files inside a folder named 'npz') found. Laptop: unzip the Kaggle "
                       "Stage 2 output into data/stage2/. Kaggle: Add Input -> Notebook Output -> your Stage 2 "
                       "notebook. Or set POSE_DIR_OVERRIDE." + LAYOUT_HELP)
for p, n in cands:
    print(f"  found {n:3d} videos in {p}")
NPZ_DIR, _ = max(cands, key=lambda c: c[1])
RUN_NAME = NPZ_DIR.parent.name
STAGE2_DIR = NPZ_DIR.parent

def find_labels():
    if LABELS_CSV_OVERRIDE:
        return Path(LABELS_CSV_OVERRIDE)
    out = []
    for p in sorted(INPUT_ROOT.rglob("*.csv")) if INPUT_ROOT.exists() else []:
        if DATASET_HINT not in p.as_posix() or in_stage2(p):
            continue
        try:
            head = next(csv.reader(open(p, newline="", encoding="utf-8")), [])
        except Exception:
            continue
        if {"video_id", "frame_idx", "x1", "y1", "x2", "y2", "status"} <= set(head):
            out.append(p)
    out.sort(key=lambda p: (p.name != "patient_labels.csv", len(p.as_posix())))
    return out[0] if out else None

def find_meta():
    if META_CSV_OVERRIDE:
        return Path(META_CSV_OVERRIDE)
    for p in sorted(INPUT_ROOT.rglob("*.csv")) if INPUT_ROOT.exists() else []:
        try:
            head = next(csv.reader(open(p, newline="", encoding="utf-8")), [])
        except Exception:
            continue
        if "video_id" in head and "layout" in head:
            return p
    return None

LABELS_CSV, META_CSV = find_labels(), find_meta()
LABELS = pd.read_csv(LABELS_CSV) if LABELS_CSV else pd.DataFrame(columns=["video_id", "frame_idx", "status"])
LABELS["video_id"] = LABELS["video_id"].astype(str)
META = pd.read_csv(META_CSV) if META_CSV else None
if META is not None:
    META["video_id"] = META["video_id"].astype(str)

SRC = {}
roots = [Path(VIDEO_DIR_OVERRIDE)] if VIDEO_DIR_OVERRIDE else [INPUT_ROOT]
for r in roots:
    for p in (r.rglob("*") if r.exists() else []):
        if p.suffix.lower() in VIDEO_EXTS and not in_stage2(p):
            SRC.setdefault(p.stem, p)

OUT_DIR = WORK_ROOT / "outputs" / "02b_analysis" / RUN_NAME
FIG_DIR = OUT_DIR / "figures"
(FIG_DIR / "videos").mkdir(parents=True, exist_ok=True)
VIDEO_IDS = sorted(p.stem for p in NPZ_DIR.glob("*.npz"))
print(f"\nAnalysing run  : {RUN_NAME}  ({len(VIDEO_IDS)} videos)  from {NPZ_DIR}")
print(f"Data folder    : {INPUT_ROOT}")
print(f"Labels         : {LABELS_CSV or 'NOT FOUND -> patient metrics skipped (put patient_labels.csv in data/labels/)'}"
      + (f"  ({LABELS['video_id'].nunique()} videos, {len(LABELS)} rows)" if LABELS_CSV else ""))
print(f"Layout CSV     : {META_CSV or 'not given (optional)'}")
print(f"Video files    : {sum(v in SRC for v in VIDEO_IDS)}/{len(VIDEO_IDS)} found (needed only for images)")
missing_v = [v for v in VIDEO_IDS if v not in SRC]
if missing_v:
    print(f"                 missing e.g. {missing_v[:3]} -> put the original videos in data/videos/ for overlay images")
print(f"Output         : {OUT_DIR}")
_done('paths')

## 3. Functions

In [ ]:
if "CELLS_DONE" not in globals():
    raise RuntimeError("Run cell 1 (Settings) first: the kernel was restarted or cells were skipped.")
_require('paths')

HALPE26 = ["nose", "left_eye", "right_eye", "left_ear", "right_ear", "left_shoulder", "right_shoulder",
           "left_elbow", "right_elbow", "left_wrist", "right_wrist", "left_hip", "right_hip",
           "left_knee", "right_knee", "left_ankle", "right_ankle", "head", "neck", "hip",
           "left_big_toe", "right_big_toe", "left_small_toe", "right_small_toe", "left_heel", "right_heel"]
LINKS = [(0, 1), (0, 2), (1, 3), (2, 4), (17, 18), (18, 19), (5, 18), (6, 18), (5, 7), (7, 9), (6, 8), (8, 10),
         (11, 19), (12, 19), (11, 13), (13, 15), (12, 14), (14, 16), (15, 20), (15, 22), (15, 24),
         (16, 21), (16, 23), (16, 25)]
ANKLES, FEET = [15, 16], [20, 21, 22, 23, 24, 25]
LOWER = [11, 12, 13, 14, 15, 16, 24, 25, 20, 21]

# ---- chart style (one blue for single series; blue / orange for two; recessive axes) ----
C_BLUE, C_ORANGE, C_AQUA = "#2a78d6", "#eb6834", "#1baf7a"
C_TEXT, C_MUTED, C_GRID, C_SURF = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
C_BAD = "#d03b3b"
plt.rcParams.update({
    "figure.facecolor": C_SURF, "axes.facecolor": C_SURF, "savefig.facecolor": C_SURF,
    "axes.edgecolor": C_GRID, "axes.labelcolor": C_MUTED, "axes.titlecolor": C_TEXT, "axes.titlesize": 12,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": C_GRID, "grid.linewidth": 0.8, "xtick.color": C_MUTED, "ytick.color": C_MUTED,
    "lines.linewidth": 2, "legend.frameon": False, "font.size": 10})

def savefig(fig, name):
    p = FIG_DIR / f"{name}.png"
    p.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(p, dpi=130, bbox_inches="tight")
    plt.show()
    return p

def load(video_id):
    with np.load(NPZ_DIR / f"{video_id}.npz") as d:
        a = {k: d[k] for k in d.files}
    a["meta"] = json.loads(str(a["meta"]))
    fp, f = a["frames_processed"], a["det_frame"]
    a["start"] = np.searchsorted(f, fp, "left")
    a["end"] = np.searchsorted(f, fp, "right")
    a["pos"] = {int(x): k for k, x in enumerate(fp)}          # frame index -> position in frames_processed
    return a

def dets_at(a, frame):
    k = a["pos"].get(int(frame))
    return (0, 0) if k is None else (int(a["start"][k]), int(a["end"][k]))

def iou_one_many(b, B):
    if len(B) == 0:
        return np.zeros(0)
    x1 = np.maximum(b[0], B[:, 0]); y1 = np.maximum(b[1], B[:, 1])
    x2 = np.minimum(b[2], B[:, 2]); y2 = np.minimum(b[3], B[:, 3])
    inter = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
    return inter / ((b[2] - b[0]) * (b[3] - b[1]) + (B[:, 2] - B[:, 0]) * (B[:, 3] - B[:, 1]) - inter + 1e-9)

def label_rows(video_id):
    return LABELS[LABELS["video_id"] == video_id]

def has_box(r):
    return pd.notna(r.get("x1")) and str(r.get("x1")).strip() != ""

def match_patient(a, r):
    """Detection index of the patient at a labelled frame (max IoU with the label box) or None."""
    s0, e0 = dets_at(a, r["frame_idx"])
    if e0 == s0 or not has_box(r):
        return None, 0.0
    ious = iou_one_many(np.array([r["x1"], r["y1"], r["x2"], r["y2"]], float), a["det_boxes"][s0:e0])
    j = int(ious.argmax())
    return s0 + j, float(ious[j])

def follow(a, j0, frame0, seconds, link_iou=LINK_IOU, max_gap=2):
    """Follow one box forward and backward by greedy max-IoU linking (a tiny local tracker).
    Returns list of (frame, det_index or None)."""
    fp = a["frames_processed"]
    k0 = a["pos"][int(frame0)]
    n = int(round(seconds * a["meta"]["fps"] / max(1, a["meta"].get("stride", 2))))
    out = {int(frame0): j0}
    for step in (1, -1):
        box, gap = a["det_boxes"][j0], 0
        for k in range(k0 + step, k0 + step * (n + 1), step):
            if k < 0 or k >= len(fp):
                break
            s0, e0 = int(a["start"][k]), int(a["end"][k])
            ious = iou_one_many(box, a["det_boxes"][s0:e0])
            if len(ious) and ious.max() >= link_iou:
                j = s0 + int(ious.argmax()); out[int(fp[k])] = j; box, gap = a["det_boxes"][j], 0
            else:
                out[int(fp[k])] = None; gap += 1
                if gap > max_gap:
                    break
    return sorted(out.items())

def draw(frame, a, frame_idx, patient_j=None, gt=None):
    img = frame.copy()
    s0, e0 = dets_at(a, frame_idx)
    for j in range(s0, e0):
        b, sc = a["det_boxes"][j].astype(int), a["det_scores"][j]
        is_p = j == patient_j
        col = (0, 200, 0) if is_p else ((0, 200, 255) if sc >= STRONG_DET else (0, 120, 160))
        cv2.rectangle(img, (b[0], b[1]), (b[2], b[3]), col, 3 if is_p else 2)
        cv2.putText(img, f"{sc:.2f}", (b[0] + 3, b[1] + 18), cv2.FONT_HERSHEY_SIMPLEX, 0.6, col, 2)
        kp, ks = a["det_kpts"][j], a["det_kpt_scores"][j]
        if np.isnan(kp).any():
            continue
        for p, q in LINKS:
            if ks[p] >= KPT_THR and ks[q] >= KPT_THR:
                cv2.line(img, tuple(kp[p].astype(int)), tuple(kp[q].astype(int)), (255, 255, 0), 2)
        for k_, (pt, s_) in enumerate(zip(kp, ks)):
            if s_ >= KPT_THR:
                cv2.circle(img, tuple(pt.astype(int)), 5 if k_ in ANKLES else 3,
                           (0, 0, 255) if k_ in ANKLES else (0, 255, 0), -1)
    if gt is not None:
        g = np.asarray(gt).astype(int)
        cv2.rectangle(img, (g[0], g[1]), (g[2], g[3]), (255, 0, 255), 2)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def read_frames(video_id, frames):
    want, got = set(int(f) for f in frames), {}
    p = SRC.get(video_id)
    if p is None or not want:
        return got
    cap, i = cv2.VideoCapture(str(p)), 0
    while len(got) < len(want) and cap.grab():          # sequential read = same frame numbers as Stage 2
        if i in want:
            ok, fr = cap.retrieve()
            if ok:
                got[i] = fr
        i += 1
    cap.release()
    return got

print("Functions ready.")
_done('functions')

## 4. Detection quality (YOLOX), per video
* `people_mean` / `pct_no_person` / `pct_multi`: boxes with score ≥ `STRONG_DET` per frame.
* `count_changes_per_min`: how often that count changes (flicker = missed or spurious boxes).
* `weak_box_share`: share of saved boxes below `STRONG_DET` (kept for ByteTrack).
* Patient (labelled frames): `recall` = IoU ≥ `IOU_MATCH`, `iou_mean`, `patient_score_mean`, and how often the
  patient is the **most central**, **highest-score** or **largest** box (a preview of the Stage 4 selection rules).

In [ ]:
if "CELLS_DONE" not in globals():
    raise RuntimeError("Run cell 1 (Settings) first: the kernel was restarted or cells were skipped.")
_require('functions')

det_rows, frame_rows = [], []
for vid in VIDEO_IDS:
    a = load(vid)
    m, sc, B = a["meta"], a["det_scores"], a["det_boxes"]
    W, H, fps = m["width"], m["height"], m["fps"]
    n_strong = np.array([(sc[s0:e0] >= STRONG_DET).sum() for s0, e0 in zip(a["start"], a["end"])])
    dur_min = len(a["frames_processed"]) * m.get("stride", 2) / fps / 60
    hfrac = (B[:, 3] - B[:, 1])[sc >= STRONG_DET] / H
    row = dict(video_id=vid, duration_s=round(dur_min * 60, 1), frames=len(a["frames_processed"]),
               width=W, height=H, fps=round(fps, 2),
               people_mean=round(float(n_strong.mean()), 2), people_max=int(n_strong.max(initial=0)),
               pct_no_person=round(float((n_strong == 0).mean()), 3),
               pct_multi=round(float((n_strong >= 2).mean()), 3),
               count_changes_per_min=round(float((np.diff(n_strong) != 0).sum() / max(dur_min, 1e-9)), 1),
               weak_box_share=round(float((sc < STRONG_DET).mean()) if len(sc) else 0.0, 3),
               box_height_median=round(float(np.median(hfrac)), 3) if len(hfrac) else np.nan)
    # ---- patient at labelled frames
    for _, r in label_rows(vid).iterrows():
        f = int(r["frame_idx"])
        rec = dict(video_id=vid, frame_idx=f, t_s=round(f / fps, 2), status=r["status"],
                   processed=int(f in a["pos"]), n_people=int(n_strong[a["pos"][f]]) if f in a["pos"] else np.nan)
        if not has_box(r):
            frame_rows.append(rec); continue
        j, iou = match_patient(a, r)
        rec.update(best_iou=round(iou, 3), detected=int(j is not None and iou >= IOU_MATCH))
        if j is not None and iou >= IOU_MATCH:
            s0, e0 = dets_at(a, f)
            strong = [k for k in range(s0, e0) if sc[k] >= STRONG_DET] or list(range(s0, e0))
            cx, cy = (B[strong, 0] + B[strong, 2]) / 2, (B[strong, 1] + B[strong, 3]) / 2
            dist = np.hypot(cx - W / 2, cy - H / 2)
            area = (B[strong, 2] - B[strong, 0]) * (B[strong, 3] - B[strong, 1])
            ks = a["det_kpt_scores"][j]
            rec.update(patient_score=round(float(sc[j]), 3),
                       patient_most_central=int(strong[int(dist.argmin())] == j),
                       patient_top_score=int(strong[int(sc[strong].argmax())] == j),
                       patient_largest=int(strong[int(area.argmax())] == j),
                       ankle_conf=round(float(ks[ANKLES].mean()), 3), feet_conf=round(float(ks[FEET].mean()), 3),
                       ankles_seen=int((ks[ANKLES] >= KPT_THR).sum()))
        frame_rows.append(rec)
    det_rows.append(row)

DET = pd.DataFrame(det_rows).set_index("video_id")
LF = pd.DataFrame(frame_rows)
if len(LF) and "detected" in LF:
    g = LF.dropna(subset=["detected"]).groupby("video_id")
    DET = DET.join(pd.DataFrame({
        "labelled_frames": g.size(), "recall": g["detected"].mean().round(3), "iou_mean": g["best_iou"].mean().round(3),
        "patient_score_mean": g["patient_score"].mean().round(3),
        "pct_patient_most_central": g["patient_most_central"].mean().round(3),
        "pct_patient_top_score": g["patient_top_score"].mean().round(3),
        "pct_patient_largest": g["patient_largest"].mean().round(3)}))

print(f"{len(DET)} videos | mean people/frame {DET['people_mean'].mean():.2f} | "
      f"frames with nobody {DET['pct_no_person'].mean():.1%}")
if "recall" in DET:
    vis = LF.dropna(subset=["detected"])
    print(f"Patient detected at {vis['detected'].mean():.1%} of {len(vis)} labelled frames "
          f"(mean IoU {vis['best_iou'].mean():.3f})")
    print("When detected, the patient is the most central box in "
          f"{vis['patient_most_central'].mean():.1%}, highest-score box in {vis['patient_top_score'].mean():.1%}, "
          f"largest box in {vis['patient_largest'].mean():.1%} of labelled frames.")
    if (LF["processed"] == 0).any():
        print(f"WARNING: {(LF['processed'] == 0).sum()} labelled frames are not in frames_processed (stride mismatch?)")
display(DET.sort_values("recall" if "recall" in DET else "pct_no_person").head(15))

# ---- figure: patient recall per video (sorted, worst at the top)
if "recall" in DET:
    d = DET["recall"].dropna().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(9, max(3, 0.22 * len(d) + 1)))
    cols = [C_BLUE if v >= FLAG_RECALL else C_BAD for v in d.values]
    ax.barh(range(len(d)), d.values, color=cols, height=0.7)
    ax.set_yticks(range(len(d))); ax.set_yticklabels(d.index, fontsize=8)
    ax.axvline(FLAG_RECALL, color=C_MUTED, lw=1, ls="--")
    ax.text(FLAG_RECALL + 0.01, 0.01, f"red = flagged (< {FLAG_RECALL:.0%})", color=C_MUTED, fontsize=8,
            va="bottom", transform=ax.get_xaxis_transform())
    ax.set_xlim(0, 1.02); ax.set_xlabel("share of labelled frames where the patient box was detected (IoU ≥ 0.5)")
    ax.set_title("Patient detection recall per video"); ax.grid(axis="y", visible=False)
    savefig(fig, "det_recall_per_video")

# ---- figure: score distribution of all saved boxes + people-per-frame distribution
all_sc = np.concatenate([load(v)["det_scores"] for v in VIDEO_IDS])
fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))
axes[0].hist(all_sc, bins=np.linspace(0, 1, 41), color=C_BLUE, rwidth=0.9)
axes[0].axvline(STRONG_DET, color=C_MUTED, lw=1, ls="--"); axes[0].set_yscale("log")
axes[0].set_xlabel("detection score"); axes[0].set_ylabel("boxes (log)")
axes[0].set_title(f"All saved boxes: {len(all_sc):,} (dashed = STRONG_DET)")
axes[1].hist(DET["people_mean"], bins=20, color=C_BLUE, rwidth=0.9)
axes[1].set_xlabel("mean people per frame (score ≥ STRONG_DET)"); axes[1].set_ylabel("videos")
axes[1].set_title("How crowded are the videos?")
savefig(fig, "det_scores_and_crowding")
_done('detection')

## 5. Pose quality (RTMPose, Halpe-26)
Mean keypoint confidence per joint for **the patient** (at labelled frames) and for **all strong boxes**.
Ankles (15, 16), heels (24, 25) and toes (20–23) drive the Stage 4 energy and IMU rules, so look there first.

In [ ]:
if "CELLS_DONE" not in globals():
    raise RuntimeError("Run cell 1 (Settings) first: the kernel was restarted or cells were skipped.")
_require('detection')

pat_scores, all_scores, per_vid_joint = [], [], {}
for vid in VIDEO_IDS:
    a = load(vid)
    keep = (a["det_scores"] >= STRONG_DET) & ~np.isnan(a["det_kpts"][:, 0, 0])
    if keep.any():
        all_scores.append(a["det_kpt_scores"][keep])
    js = []
    for _, r in label_rows(vid).iterrows():
        j, iou = match_patient(a, r)
        if j is not None and iou >= IOU_MATCH and not np.isnan(a["det_kpts"][j, 0, 0]):
            js.append(a["det_kpt_scores"][j])
    if js:
        pat_scores += js
        per_vid_joint[vid] = np.mean(js, axis=0)

JOINT = pd.DataFrame({"joint": HALPE26,
                      "patient_mean": np.mean(pat_scores, axis=0) if pat_scores else np.nan,
                      "all_people_mean": np.concatenate(all_scores).mean(0) if all_scores else np.nan,
                      "patient_pct_seen": (np.array(pat_scores) >= KPT_THR).mean(0) if pat_scores else np.nan})
JOINT.index.name = "kpt_idx"
POSE = pd.DataFrame({vid: {"ankle_conf": v[ANKLES].mean(), "heel_conf": v[[24, 25]].mean(),
                           "toe_conf": v[[20, 21, 22, 23]].mean(), "knee_conf": v[[13, 14]].mean(),
                           "hip_conf": v[[11, 12]].mean()} for vid, v in per_vid_joint.items()}).T.round(3)
display(JOINT.round(3))

# ---- figure: per-joint confidence, patient vs everyone (dot plot)
order = JOINT.sort_values("all_people_mean").index
fig, ax = plt.subplots(figsize=(9, 7))
y = np.arange(len(order))
ax.hlines(y, JOINT.loc[order, ["patient_mean", "all_people_mean"]].min(axis=1),
          JOINT.loc[order, ["patient_mean", "all_people_mean"]].max(axis=1), color=C_GRID, lw=2)
ax.scatter(JOINT.loc[order, "all_people_mean"], y, s=48, color=C_ORANGE, label="all strong boxes", zorder=3,
           edgecolor=C_SURF, linewidth=2)
ax.scatter(JOINT.loc[order, "patient_mean"], y, s=48, color=C_BLUE, label="patient (labelled frames)", zorder=4,
           edgecolor=C_SURF, linewidth=2)
ax.set_yticks(y); ax.set_yticklabels([f"{i:2d} {JOINT.loc[i, 'joint']}" for i in order], fontsize=8)
for t in ax.get_yticklabels():
    if int(t.get_text().split()[0]) in ANKLES + FEET:
        t.set_color(C_TEXT); t.set_fontweight("bold")
ax.axvline(KPT_THR, color=C_MUTED, lw=1, ls="--")
ax.set_xlim(0, 1); ax.set_xlabel("mean keypoint confidence (dashed = KPT_THR)")
ax.set_title("Which joints does RTMPose see? (bold = ankles / feet)"); ax.grid(axis="y", visible=False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.09), ncol=2)
savefig(fig, "pose_joint_confidence")

# ---- figure: lower-body confidence per video (heatmap, one blue ramp)
if len(per_vid_joint):
    Hm = pd.DataFrame({v: s[LOWER] for v, s in per_vid_joint.items()}, index=[HALPE26[i] for i in LOWER]).T
    Hm = Hm.loc[Hm.iloc[:, 4:6].mean(axis=1).sort_values().index]              # sort by ankle confidence
    from matplotlib.colors import LinearSegmentedColormap
    cmap = LinearSegmentedColormap.from_list("blue", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
    fig, ax = plt.subplots(figsize=(9, max(3, 0.22 * len(Hm) + 1.5)))
    im = ax.imshow(Hm.values, aspect="auto", cmap=cmap, vmin=0, vmax=1)
    ax.set_xticks(range(len(Hm.columns))); ax.set_xticklabels(Hm.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(Hm))); ax.set_yticklabels(Hm.index, fontsize=8); ax.grid(False)
    fig.colorbar(im, ax=ax, fraction=0.03, label="patient keypoint confidence")
    ax.set_title("Patient lower-body keypoint confidence per video (worst ankles at the top)")
    savefig(fig, "pose_lower_body_heatmap")
_done('pose')

## 6. Following the patient ±2 s around each labelled frame
A tiny IoU tracker starts at the patient box of each labelled frame and follows it through the neighbouring
processed frames. Per snippet:
* `coverage`: share of frames where the box could be linked (low = the patient disappears / is mixed up),
* `ankle_seen`: share of linked frames where both ankles have score ≥ `KPT_THR`,
* `ankle_jitter`: median |second difference| of the ankle position divided by box height. Real walking also
  contributes, so compare between videos; a much higher value means noisy keypoints.

In [ ]:
if "CELLS_DONE" not in globals():
    raise RuntimeError("Run cell 1 (Settings) first: the kernel was restarted or cells were skipped.")
_require('pose')

snip_rows, SNIPPET_SERIES = [], {}
for vid in VIDEO_IDS:
    a = load(vid)
    for _, r in label_rows(vid).iterrows():
        j0, iou = match_patient(a, r)
        if j0 is None or iou < IOU_MATCH:
            continue
        chain = follow(a, j0, int(r["frame_idx"]), SNIPPET_S)
        frames = np.array([f for f, _ in chain]); js = [j for _, j in chain]
        linked = np.array([j is not None for j in js])
        P = np.full((len(js), 2, 2), np.nan); C = np.zeros((len(js), 2)); Hh = np.full(len(js), np.nan)
        for k, j in enumerate(js):
            if j is None or np.isnan(a["det_kpts"][j, 0, 0]):
                continue
            P[k] = a["det_kpts"][j, ANKLES]; C[k] = a["det_kpt_scores"][j, ANKLES]
            Hh[k] = a["det_boxes"][j, 3] - a["det_boxes"][j, 1]
        P[C < KPT_THR] = np.nan
        acc = np.linalg.norm(P[2:] - 2 * P[1:-1] + P[:-2], axis=2) / Hh[1:-1, None] if len(js) >= 3 else np.zeros((0, 2))
        both = (C >= KPT_THR).all(1)
        snip_rows.append(dict(video_id=vid, frame_idx=int(r["frame_idx"]), frames=len(js),
                              coverage=round(float(linked.mean()), 3),
                              ankle_seen=round(float(both[linked].mean()) if linked.any() else 0.0, 3),
                              ankle_jitter=round(float(np.nanmedian(acc)), 4) if np.isfinite(acc).any() else np.nan))
        SNIPPET_SERIES[(vid, int(r["frame_idx"]))] = dict(frames=frames, P=P, C=C, H=Hh, fps=a["meta"]["fps"])
SNIP = pd.DataFrame(snip_rows)
if len(SNIP):
    SN = SNIP.groupby("video_id")[["coverage", "ankle_seen", "ankle_jitter"]].mean().round(4)
    SN.columns = ["snippet_coverage", "snippet_ankle_seen", "snippet_ankle_jitter"]
    print(f"{len(SNIP)} snippets | mean coverage {SNIP['coverage'].mean():.1%} | both ankles seen "
          f"{SNIP['ankle_seen'].mean():.1%} | median jitter {SNIP['ankle_jitter'].median():.4f}")
    fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))
    axes[0].hist(SNIP["coverage"], bins=np.linspace(0, 1, 21), color=C_BLUE, rwidth=0.9)
    axes[0].axvline(FLAG_COVERAGE, color=C_MUTED, ls="--", lw=1)
    axes[0].set_xlabel("share of snippet frames where the patient box could be linked"); axes[0].set_ylabel("snippets")
    axes[0].set_title("Can the patient be followed for ±2 s?")
    axes[1].hist(SNIP["ankle_jitter"].dropna(), bins=30, color=C_BLUE, rwidth=0.9)
    axes[1].set_xlabel("ankle jitter (|2nd difference| / box height, median per snippet)"); axes[1].set_ylabel("snippets")
    axes[1].set_title("How noisy are the ankle keypoints?")
    savefig(fig, "snippets_coverage_jitter")
else:
    SN = pd.DataFrame()
    print("No snippets (no labels or no matched patient boxes).")
_done('snippets')

## 7. One video in detail
Set `VIEW_VIDEO` in cell 1 (or here) and re-run only this cell. Shows: people per frame over time, every box score
over time, the patient's ankle trajectory and confidence in each snippet, and overlay images of the labelled frames
(green = matched patient box, magenta = your label, red dots = ankles).

In [ ]:
if "CELLS_DONE" not in globals():
    raise RuntimeError("Run cell 1 (Settings) first: the kernel was restarted or cells were skipped.")
_require('snippets')

vid = VIEW_VIDEO or (DET["recall"].idxmin() if "recall" in DET and DET["recall"].notna().any() else VIDEO_IDS[0])
a = load(vid); m = a["meta"]; fps = m["fps"]
t = a["frames_processed"] / fps
sc = a["det_scores"]
print(f"=== {vid} ===")
card = DET.loc[[vid]].join(POSE, how="left").join(SN, how="left").T
card.columns = [vid]
display(card)

fig, axes = plt.subplots(2, 1, figsize=(16, 5.5), sharex=True)
n_all = a["end"] - a["start"]
n_strong = np.array([(sc[s0:e0] >= STRONG_DET).sum() for s0, e0 in zip(a["start"], a["end"])])
axes[0].plot(t, n_all, color=C_ORANGE, lw=1.2, label="all saved boxes")
axes[0].plot(t, n_strong, color=C_BLUE, lw=2, label=f"score ≥ {STRONG_DET}")
axes[0].set_ylabel("people"); axes[0].legend(loc="upper right"); axes[0].set_title(f"{vid}: people per frame")
axes[1].scatter(a["det_frame"] / fps, sc, s=2, color=C_BLUE, alpha=0.35)
axes[1].axhline(STRONG_DET, color=C_MUTED, ls="--", lw=1)
axes[1].set_ylabel("box score"); axes[1].set_xlabel("time (s)"); axes[1].set_title("every box score over time")
for f in label_rows(vid)["frame_idx"]:
    for ax in axes:
        ax.axvline(f / fps, color=C_MUTED, alpha=0.5, lw=1, ls=":")
savefig(fig, f"videos/{vid}_timeline")

keys = [k for k in SNIPPET_SERIES if k[0] == vid]
if keys:
    fig, axes = plt.subplots(2, 1, figsize=(16, 5.5), sharex=True)
    for n_, k in enumerate(keys):
        s = SNIPPET_SERIES[k]; tt = s["frames"] / s["fps"]
        for side, col, nm in ((0, C_BLUE, "left ankle"), (1, C_ORANGE, "right ankle")):
            y = (s["P"][:, side, 1] - np.nanmean(s["P"][:, side, 1])) / np.nanmean(s["H"])
            axes[0].plot(tt, y, color=col, lw=1.5, label=nm if n_ == 0 else None)
            axes[1].plot(tt, s["C"][:, side], color=col, lw=1.5, label=nm if n_ == 0 else None)
    axes[0].set_ylabel("ankle y / box height\n(centred per snippet)"); axes[0].invert_yaxis()
    axes[0].set_title("patient ankle height in each ±2 s snippet (gaps = keypoint below KPT_THR or box lost)")
    axes[1].axhline(KPT_THR, color=C_MUTED, ls="--", lw=1)
    axes[1].set_ylabel("ankle confidence"); axes[1].set_xlabel("time (s)"); axes[1].set_ylim(0, 1)
    axes[0].legend(loc="upper right")
    savefig(fig, f"videos/{vid}_ankles")

rows = label_rows(vid)
frames = read_frames(vid, rows["frame_idx"].astype(int).tolist())
if frames:
    cols = 4; nrow = (len(rows) + cols - 1) // cols
    fig, axes = plt.subplots(nrow, cols, figsize=(18, 3.8 * nrow), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (_, r) in zip(axes.ravel(), rows.iterrows()):
        f = int(r["frame_idx"])
        if f not in frames:
            continue
        j, iou = match_patient(a, r)
        gt = [r["x1"], r["y1"], r["x2"], r["y2"]] if has_box(r) else None
        ax.imshow(draw(frames[f], a, f, j if iou >= IOU_MATCH else None, gt))
        ok = "detected" if (j is not None and iou >= IOU_MATCH) else ("not visible" if gt is None else "MISSED")
        ax.set_title(f"frame {f} (t={f / fps:.1f}s)  IoU {iou:.2f}  {ok}", fontsize=9,
                     color=C_BAD if ok == "MISSED" else C_TEXT)
    plt.tight_layout()
    savefig(fig, f"videos/{vid}_labelled_frames")
else:
    print(f"Original video for {vid} not found in {INPUT_ROOT} -> overlay images skipped "
          "(laptop: put the videos in data/videos/; Kaggle: add the videos dataset as input).")
_done('video')

## 8. Videos that need attention

In [ ]:
if "CELLS_DONE" not in globals():
    raise RuntimeError("Run cell 1 (Settings) first: the kernel was restarted or cells were skipped.")
_require('snippets')

SUMMARY = DET.join(POSE, how="left").join(SN, how="left")
if META is not None:
    SUMMARY = SUMMARY.join(META.set_index("video_id"), how="left")
def reasons(r):
    out = []
    if pd.notna(r.get("recall")) and r["recall"] < FLAG_RECALL:
        out.append(f"patient detected in only {r['recall']:.0%} of labelled frames")
    if pd.notna(r.get("ankle_conf")) and r["ankle_conf"] < FLAG_ANKLE_CONF:
        out.append(f"low patient ankle confidence ({r['ankle_conf']:.2f})")
    if r["pct_no_person"] > FLAG_NO_PERSON:
        out.append(f"{r['pct_no_person']:.0%} of frames without any person")
    if pd.notna(r.get("snippet_coverage")) and r["snippet_coverage"] < FLAG_COVERAGE:
        out.append(f"patient box lost in snippets (coverage {r['snippet_coverage']:.0%})")
    if pd.notna(r.get("pct_patient_most_central")) and r["pct_patient_most_central"] < 0.5:
        out.append("patient usually NOT the most central person (position rule will struggle)")
    return "; ".join(out)
SUMMARY["flags"] = SUMMARY.apply(reasons, axis=1)
FLAGS = SUMMARY[SUMMARY["flags"] != ""][["flags"]]
print(f"{len(FLAGS)} of {len(SUMMARY)} videos flagged.")
display(FLAGS)
if META is not None:
    grp = [c for c in META.columns if c != "video_id"][0]
    cols = [c for c in ("recall", "iou_mean", "people_mean", "pct_no_person", "ankle_conf", "snippet_coverage",
                        "snippet_ankle_jitter", "pct_patient_most_central") if c in SUMMARY]
    print(f"\nBy {grp}:")
    display(SUMMARY.groupby(grp)[cols].mean().round(3).assign(videos=SUMMARY.groupby(grp).size()))
_done('flags')

## 9. Save everything

In [ ]:
if "CELLS_DONE" not in globals():
    raise RuntimeError("Run cell 1 (Settings) first: the kernel was restarted or cells were skipped.")
_require('flags')

SUMMARY.to_csv(OUT_DIR / "per_video_summary.csv")
LF.to_csv(OUT_DIR / "labelled_frames.csv", index=False)
JOINT.to_csv(OUT_DIR / "joint_confidence.csv")
SNIP.to_csv(OUT_DIR / "snippets.csv", index=False)
FLAGS.to_csv(OUT_DIR / "flags.csv")
(OUT_DIR / "config.json").write_text(json.dumps(dict(
    stage2_run=RUN_NAME, stage2_npz=str(NPZ_DIR), labels_csv=str(LABELS_CSV), meta_csv=str(META_CSV),
    STRONG_DET=STRONG_DET, IOU_MATCH=IOU_MATCH, KPT_THR=KPT_THR, SNIPPET_S=SNIPPET_S, LINK_IOU=LINK_IOU,
    created=datetime.now().isoformat(timespec="seconds")), indent=2))
idx = WORK_ROOT / "outputs" / "index.csv"
rows = [r for r in csv.DictReader(open(idx, newline="", encoding="utf-8"))
        if not (r["stage"] == "02b_analysis" and r["run_name"] == RUN_NAME)] if idx.exists() else []
rows.append(dict(stage="02b_analysis", run_name=RUN_NAME, path=f"02b_analysis/{RUN_NAME}",
                 videos_done=len(SUMMARY), videos_total=len(VIDEO_IDS),
                 updated=datetime.now().isoformat(timespec="seconds"),
                 notes=json.dumps(dict(flagged=len(FLAGS), recall=float(LF["detected"].mean()) if "detected" in LF else None))))
fields = list(dict.fromkeys(k for r in rows for k in r))
with open(idx, "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=fields); w.writeheader(); w.writerows(rows)
print("Saved to", OUT_DIR)
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(OUT_DIR))
_done('save')

---
# Part B — Who is the patient, who is an observer, and where might the label have switched?

**How the notebook decides who the patient is**
1. A **tracker** links the Stage 2 boxes over time into tracks (one ID per person): `bytetrack` (Kalman filter +
   two-stage matching, default) or `iou` (greedy IoU, the simple baseline).
2. Your **8 labelled frames are anchors**: at each one, the track that covers your label box is the patient.
3. Between two anchors the **patient label follows that track**. Every other track is an **observer**.
4. If anchor *k* says "patient = track A" and anchor *k+1* says "patient = track B", while A and B exist **at the same
   time**, the label must have moved between two people somewhere in between → **`anchor_conflict`**
   (strongest evidence). The notebook picks the switch frame where the patient's path is most continuous.

**Suspected switch events** (written to `identity/events.csv`, reviewed with `review_patient.py`):

| type | meaning | how suspicious |
|---|---|---|
| `anchor_conflict` | two different "patient" tracks exist at the same time between two anchors | very high |
| `anchor_mismatch` | at a labelled frame the patient label sits on a box that is NOT your label box | high |
| `handover_far` | patient track ends and the next patient track starts far away | medium |
| `jump` | patient box jumps more than `JUMP_FRAC` × its height in one step | medium |
| `close_encounter` | patient box overlaps / comes very close to another person (crossing, passing) | depends on overlap |
| `handover` | patient track ends and a new one continues nearby (fragmentation, usually harmless) | low |

**Then watch the videos and mark anomalies by hand:** in a terminal in your project folder run
`python review_patient.py`. It plays each video with PASIEN / OBS boxes; press `1`–`4` when an anomaly starts and
again when it ends (1 = label moved to an observer, 2 = observer fills the patient box, 3 = patient missing,
4 = other), and `o` when a video is done. **Cell 14** turns your marks into statistics, features that describe each
anomaly type (to design a fix), and corrected patient tracks.

```
outputs/02b_analysis/<RUN_NAME>/identity/
├── identity_summary.csv        ← per video: tracks, patient coverage, anchors, conflicts, suspects
├── events.csv                  ← suspected switches (input for review_patient.py)
├── anchors.csv                 ← your 8 labelled frames per video and the track each one matched
├── tracks/<video_id>.csv       ← EVERY tracked box with its role: patient / observer
├── patient/<video_id>.csv      ← one row per processed frame: patient track + box (or none)
├── anomaly_review.csv          ← review_patient.py: one row per anomaly you marked (type, start, end, tracks)
├── video_review.csv            ← review_patient.py: one row per video (ok / ada_anomali / belum_selesai)
├── review_summary.csv          ← cell 14: anomalies per video
├── anomaly_features.csv        ← cell 14: box features inside each anomaly (input for a solution)
├── anomaly_feature_summary.csv ← cell 14: features per anomaly type vs normal frames
└── patient_corrected/<video_id>.csv   ← cell 14: patient track after your corrections + quality per frame
```

## 10. Part B settings

In [ ]:
if "CELLS_DONE" not in globals():
    raise RuntimeError("Run cell 1 (Settings) first: the kernel was restarted or cells were skipped.")
_require('functions')

TRACKER     = "bytetrack"  # "bytetrack" (Kalman + 2-stage matching) or "iou" (greedy IoU baseline)
TRACK_HIGH  = 0.5     # boxes with score >= this start / continue tracks
TRACK_LOW   = 0.1     # ByteTrack 2nd matching also uses boxes in [TRACK_LOW, TRACK_HIGH)
TRACK_NEW   = 0.6     # ByteTrack: minimum score to start a new track
TRACK_IOU   = 0.3     # minimum IoU to continue a track
MAX_AGE_S   = 1.0     # a lost track is kept this long (s) before it is deleted
ENC_IOU     = 0.15    # close encounter: patient box overlaps another person's box by at least this IoU ...
ENC_DIST    = 0.6     # ... or the box centres are closer than this x patient box height
JUMP_FRAC   = 0.5     # jump: patient box centre moves more than this x box height in one processed frame
MERGE_S     = 1.0     # suspected events closer than this (s) are merged into one event
SUSPECT_MIN = 2.0     # events with priority >= this count as "suspects" in the summary
ID_VIEW_VIDEO = None  # cell 13: video to show; None = the video with most suspects

ID_DIR = OUT_DIR / "identity"
for sub in ("tracks", "patient", "patient_corrected"):
    (ID_DIR / sub).mkdir(parents=True, exist_ok=True)
C_PAT, C_OBS = "#1baf7a", "#eb6834"                      # patient / observer (charts)
BGR_PAT, BGR_OBS, BGR_OTHER = (122, 175, 27), (52, 104, 235), (150, 150, 150)
print("Part B output:", ID_DIR)
_done('id-settings')

## 11. Trackers (ByteTrack and greedy IoU)

In [ ]:
if "CELLS_DONE" not in globals():
    raise RuntimeError("Run cell 1 (Settings) first: the kernel was restarted or cells were skipped.")
_require('id-settings')

try:
    from scipy.optimize import linear_sum_assignment
except ImportError:
    linear_sum_assignment = None
    print("scipy not installed -> ByteTrack falls back to greedy matching. Recommended: pip install scipy")

def iou_matrix(A, Bx):
    A = np.asarray(A, float).reshape(-1, 4); Bx = np.asarray(Bx, float).reshape(-1, 4)
    if len(A) == 0 or len(Bx) == 0:
        return np.zeros((len(A), len(Bx)))
    x1 = np.maximum(A[:, None, 0], Bx[None, :, 0]); y1 = np.maximum(A[:, None, 1], Bx[None, :, 1])
    x2 = np.minimum(A[:, None, 2], Bx[None, :, 2]); y2 = np.minimum(A[:, None, 3], Bx[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (A[:, 2] - A[:, 0]) * (A[:, 3] - A[:, 1]); bb = (Bx[:, 2] - Bx[:, 0]) * (Bx[:, 3] - Bx[:, 1])
    return inter / (aa[:, None] + bb[None, :] - inter + 1e-9)

def match(iou, thr, greedy=False):
    """(row, col) pairs with IoU >= thr. Hungarian (scipy) unless greedy=True."""
    if iou.size == 0:
        return []
    if not greedy and linear_sum_assignment is not None:
        r, c = linear_sum_assignment(-iou)
        return [(int(i), int(j)) for i, j in zip(r, c) if iou[i, j] >= thr]
    out, ur, uc = [], set(), set()
    for flat in np.argsort(-iou, axis=None, kind="stable"):
        i, j = divmod(int(flat), iou.shape[1])
        if iou[i, j] < thr:
            break
        if i not in ur and j not in uc:
            out.append((i, j)); ur.add(i); uc.add(j)
    return out

def xyxy_to_xyah(b):
    w, h = b[2] - b[0], b[3] - b[1]
    return np.array([b[0] + w / 2, b[1] + h / 2, w / max(h, 1e-6), h], float)

def xyah_to_xyxy(m):
    cx, cy, a_, h = m[:4]; w = a_ * h
    return np.array([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2])

class Kalman:
    """Constant-velocity Kalman filter on (cx, cy, aspect, height), as in SORT / ByteTrack."""
    F = np.eye(8); F[:4, 4:] = np.eye(4)
    Hm = np.eye(4, 8)
    SP, SV = 1 / 20, 1 / 160
    def initiate(self, z):
        h = z[3]
        std = np.array([2 * self.SP * h, 2 * self.SP * h, 1e-2, 2 * self.SP * h,
                        10 * self.SV * h, 10 * self.SV * h, 1e-5, 10 * self.SV * h])
        return np.r_[z, np.zeros(4)], np.diag(std ** 2)
    def predict(self, m, P):
        h = m[3]
        std = np.array([self.SP * h, self.SP * h, 1e-2, self.SP * h, self.SV * h, self.SV * h, 1e-5, self.SV * h])
        return self.F @ m, self.F @ P @ self.F.T + np.diag(std ** 2)
    def update(self, m, P, z):
        h = m[3]
        R = np.diag(np.array([self.SP * h, self.SP * h, 1e-1, self.SP * h]) ** 2)
        S = self.Hm @ P @ self.Hm.T + R
        K = np.linalg.solve(S, self.Hm @ P).T
        return m + K @ (z - self.Hm @ m), P - K @ S @ K.T
KF = Kalman()

class _Track:
    def __init__(self, tid, box, k, det, confirmed):
        self.tid, self.miss, self.hits, self.confirmed = tid, 0, 1, confirmed
        self.m, self.P = KF.initiate(xyxy_to_xyah(box))
        self.pending = [] if confirmed else [(k, det)]   # first hit is written once the track is confirmed
    def box(self):
        return xyah_to_xyxy(self.m)

def run_bytetrack(a, max_age):
    tracks, next_id, out = [], 1, []
    for k in range(len(a["frames_processed"])):
        s0, e0 = int(a["start"][k]), int(a["end"][k])
        B, S = a["det_boxes"][s0:e0], a["det_scores"][s0:e0]
        for t in tracks:
            t.m, t.P = KF.predict(t.m, t.P)
        hi = [d for d in range(len(S)) if S[d] >= TRACK_HIGH]
        lo = [d for d in range(len(S)) if TRACK_LOW <= S[d] < TRACK_HIGH]
        conf = [t for t in tracks if t.confirmed]
        tent = [t for t in tracks if not t.confirmed]
        p1 = match(iou_matrix([t.box() for t in conf], B[hi]), TRACK_IOU)           # 1) confirmed vs strong
        hits = [(conf[i], hi[j]) for i, j in p1]
        used, done = {hi[j] for _, j in p1}, {id(conf[i]) for i, _ in p1}
        r2 = [t for t in conf if id(t) not in done and t.miss == 0]                 # 2) still-tracked vs weak
        p2 = match(iou_matrix([t.box() for t in r2], B[lo]), 0.5)
        hits += [(r2[i], lo[j]) for i, j in p2]; done |= {id(r2[i]) for i, _ in p2}
        rem = [d for d in hi if d not in used]                                      # 3) new (tentative) vs rest
        p3 = match(iou_matrix([t.box() for t in tent], B[rem]), TRACK_IOU)
        hits += [(tent[i], rem[j]) for i, j in p3]; done |= {id(tent[i]) for i, _ in p3}
        used |= {rem[j] for _, j in p3}
        for t, d in hits:
            t.m, t.P = KF.update(t.m, t.P, xyxy_to_xyah(B[d])); t.miss = 0; t.hits += 1
            if not t.confirmed and t.hits >= 2:
                t.confirmed = True
                out += [(kk, dd, t.tid) for kk, dd in t.pending]; t.pending = []
            if t.confirmed:
                out.append((k, s0 + d, t.tid))
            else:
                t.pending.append((k, s0 + d))
        keep = []
        for t in tracks:
            if id(t) in done:
                keep.append(t)
            elif t.confirmed:
                t.miss += 1
                if t.miss <= max_age:
                    keep.append(t)
        tracks = keep
        for d in hi:
            if d not in used and S[d] >= TRACK_NEW:
                tracks.append(_Track(next_id, B[d], k, s0 + d, confirmed=(k == 0)))
                if k == 0:
                    out.append((k, s0 + d, next_id))
                next_id += 1
    return out

def run_iou_tracker(a, max_age):
    tracks, next_id, out = [], 1, []          # each track: [id, last box, frames missed]
    for k in range(len(a["frames_processed"])):
        s0, e0 = int(a["start"][k]), int(a["end"][k])
        B, S = a["det_boxes"][s0:e0], a["det_scores"][s0:e0]
        hi = [d for d in range(len(S)) if S[d] >= TRACK_HIGH]
        pairs = match(iou_matrix([t[1] for t in tracks], B[hi]), TRACK_IOU, greedy=True)
        for i, j in pairs:
            tracks[i][1], tracks[i][2] = B[hi[j]], -1
            out.append((k, s0 + hi[j], tracks[i][0]))
        used = {hi[j] for _, j in pairs}
        for t in tracks:
            t[2] += 1
        tracks = [t for t in tracks if t[2] <= max_age]
        for d in hi:
            if d not in used:
                tracks.append([next_id, B[d], 0]); out.append((k, s0 + d, next_id)); next_id += 1
    return out

def track_video(a):
    fps_eff = a["meta"]["fps"] / max(1, a["meta"].get("stride", 2))
    max_age = max(1, int(round(MAX_AGE_S * fps_eff)))
    out = (run_bytetrack if TRACKER == "bytetrack" else run_iou_tracker)(a, max_age)
    cols = ["pos", "frame_idx", "det_idx", "track_id", "score", "x1", "y1", "x2", "y2"]
    if not out:
        return pd.DataFrame(columns=cols)
    k, d, t = (np.array(x) for x in zip(*out))
    b = a["det_boxes"][d]
    T = pd.DataFrame({"pos": k, "frame_idx": a["frames_processed"][k], "det_idx": d, "track_id": t,
                      "score": a["det_scores"][d].round(3), "x1": b[:, 0].round(1), "y1": b[:, 1].round(1),
                      "x2": b[:, 2].round(1), "y2": b[:, 3].round(1)})
    return T.sort_values(["pos", "track_id"]).reset_index(drop=True)

if TRACKER not in ("bytetrack", "iou"):
    raise ValueError('TRACKER must be "bytetrack" or "iou"')
print(f"Tracker: {TRACKER}" + ("" if linear_sum_assignment is not None or TRACKER == "iou" else " (greedy fallback)"))
_done('tracker')

## 12. Patient vs observer for every video + suspected switches
Runs the tracker on every video, assigns the patient from your anchors, marks every other track as observer, and
lists suspected switch events. Takes about a minute for 73 videos (no video files needed).

In [ ]:
if "CELLS_DONE" not in globals():
    raise RuntimeError("Run cell 1 (Settings) first: the kernel was restarted or cells were skipped.")
_require('tracker')

PRIORITY = {"anchor_conflict": 5.0, "anchor_mismatch": 4.0, "handover_far": 3.0, "jump": 2.5,
            "close_encounter": 1.0, "handover": 0.5}

def anchors_for(a, T, vid):
    det2tid = dict(zip(T["det_idx"].to_numpy(), T["track_id"].to_numpy()))
    out = []
    for _, r in label_rows(vid).iterrows():
        f = int(r["frame_idx"]); k = a["pos"].get(f)
        an = dict(video_id=vid, frame_idx=f, pos=k, status=r["status"], has_box=has_box(r),
                  x1=r.get("x1"), y1=r.get("y1"), x2=r.get("x2"), y2=r.get("y2"),
                  det_idx=-1, iou=0.0, track_id=-1, kind="")
        if k is None:
            an["kind"] = "not_processed"
        elif not an["has_box"]:
            an["kind"] = "absent"
        else:
            j, iou = match_patient(a, r)
            an["iou"] = round(iou, 3)
            if j is not None and iou >= IOU_MATCH:
                an["det_idx"], an["track_id"] = int(j), int(det2tid.get(j, -1))
                an["kind"] = "tracked" if an["track_id"] >= 0 else "detected_not_tracked"
            else:
                an["kind"] = "not_detected"
        out.append(an)
    return sorted(out, key=lambda x: x["frame_idx"])

def assign_patient(a, T, anchors):
    """Returns patient track per processed frame (-1 = none), its source, raw events, and lookup tables."""
    n = len(a["frames_processed"]); Bx = a["det_boxes"]
    alive, det_at, by_pos = {}, {}, {}
    for p, d, t in zip(T["pos"].to_numpy(), T["det_idx"].to_numpy(), T["track_id"].to_numpy()):
        alive.setdefault(int(t), np.zeros(n, bool))[p] = True
        det_at[(int(t), int(p))] = int(d)
        by_pos.setdefault(int(p), []).append((int(t), int(d)))
    def ctr(t, p):
        b = Bx[det_at[(t, p)]]
        return np.array([(b[0] + b[2]) / 2, (b[1] + b[3]) / 2]), float(b[3] - b[1])
    patient = np.full(n, -1); source = np.full(n, "", dtype=object); events = []
    def paint(t, k0, k1, src):
        for p in range(max(0, k0), min(n - 1, k1) + 1):
            if alive[t][p]:
                patient[p] = t; source[p] = src
    valid = [an for an in anchors if an["kind"] == "tracked"]
    if valid:
        paint(valid[0]["track_id"], 0, valid[0]["pos"], "anchor_track")
        paint(valid[-1]["track_id"], valid[-1]["pos"], n - 1, "anchor_track")
    for ai, aj in zip(valid[:-1], valid[1:]):
        ti, tj, ki, kj = ai["track_id"], aj["track_id"], ai["pos"], aj["pos"]
        if ti == tj:
            paint(ti, ki, kj, "anchor_track"); continue
        if not (alive[ti][ki:kj + 1] & alive[tj][ki:kj + 1]).any():
            paint(ti, ki, kj, "anchor_track"); paint(tj, ki, kj, "anchor_track")   # plain handover
            continue
        # both "patient" tracks exist at the same time -> the label moved between two people.
        # switch position s = where the patient path (ti before s, tj from s) is most continuous
        L = kj - ki + 1
        last_i = np.full(L + 1, -1); cur = -1
        for off in range(L):
            if alive[ti][ki + off]:
                cur = ki + off
            last_i[off + 1] = cur
        first_j = np.full(L + 1, -1); cur = -1
        for off in range(L - 1, -1, -1):
            if alive[tj][ki + off]:
                cur = ki + off
            first_j[off] = cur
        best, s_best = np.inf, kj
        for off in range(1, L):
            p, q = last_i[off], first_j[off]
            if p < 0 or q < 0:
                continue
            (ci, hi_), (cj, hj) = ctr(ti, p), ctr(tj, q)
            cost = np.linalg.norm(ci - cj) / ((hi_ + hj) / 2) + 0.1 * max(0, q - p - 1)   # jump + lost frames
            if cost < best - 1e-9:
                best, s_best = cost, ki + off
        paint(ti, ki, s_best - 1, "anchor_split"); paint(tj, s_best, kj, "anchor_split")
        detail = (f"track {ti} = patient at labelled frame {ai['frame_idx']}, track {tj} = patient at "
                  f"labelled frame {aj['frame_idx']}, and both exist at the same time")
        if alive[ti][kj]:
            detail += f"; at frame {aj['frame_idx']} track {ti} is on another person"
        events.append(dict(pos=s_best, start=s_best, end=s_best, type="anchor_conflict", other=ti,
                           max_iou=np.nan, detail=detail))
    # labelled frames whose label box is not where the patient label ended up
    for an in anchors:
        p = an["pos"]
        if p is None or an["kind"] in ("tracked", "not_processed") or patient[p] < 0:
            continue
        if an["kind"] == "absent":
            detail = f"label says the patient is not visible, but track {patient[p]} is marked patient"
        else:
            iou = iou_one_many(np.array([an["x1"], an["y1"], an["x2"], an["y2"]], float),
                               Bx[det_at[(int(patient[p]), p)]][None])[0]
            if iou >= IOU_MATCH:
                continue
            detail = f"at labelled frame {an['frame_idx']} the patient label (track {patient[p]}) is on another box (IoU {iou:.2f})"
        events.append(dict(pos=p, start=p, end=p, type="anchor_mismatch", other=-1, max_iou=np.nan, detail=detail))
    # handovers and jumps along the patient path
    split_pos = {e["pos"] for e in events if e["type"] == "anchor_conflict"}
    prev = None
    for p in range(n):
        t = int(patient[p])
        if t < 0:
            continue
        if prev is not None:
            tp = int(patient[prev])
            (c0, h0), (c1, h1) = ctr(tp, prev), ctr(t, p)
            dist = np.linalg.norm(c1 - c0) / ((h0 + h1) / 2)
            if t != tp and p not in split_pos:
                events.append(dict(pos=p, start=prev, end=p, type="handover_far" if dist > 1.0 else "handover",
                                   other=tp, max_iou=np.nan,
                                   detail=f"patient track {tp} ends, track {t} continues ({dist:.2f} box heights away)"))
            elif t == tp and dist / (p - prev) > JUMP_FRAC:
                events.append(dict(pos=p, start=prev, end=p, type="jump", other=-1, max_iou=np.nan,
                                   detail=f"patient box moved {dist:.2f} box heights in {p - prev} step(s)"))
        prev = p
    # close encounters with another tracked person
    enc = []
    for p in range(n):
        t = int(patient[p])
        if t < 0:
            continue
        pb = Bx[det_at[(t, p)]]; pc, ph = ctr(t, p)
        best = None
        for ot, od in by_pos.get(p, []):
            if ot == t:
                continue
            ob = Bx[od]
            iou = float(iou_one_many(pb, ob[None])[0])
            dist = float(np.linalg.norm(np.array([(ob[0] + ob[2]) / 2, (ob[1] + ob[3]) / 2]) - pc) / ph)
            if (iou >= ENC_IOU or dist < ENC_DIST) and (best is None or iou > best[0] or (iou == best[0] and dist < best[2])):
                best = (iou, ot, dist)
        if best:
            enc.append((p,) + best)
    groups, cur = [], []
    for e in enc:
        if cur and e[0] - cur[-1][0] > 2:
            groups.append(cur); cur = []
        cur.append(e)
    if cur:
        groups.append(cur)
    for g in groups:
        top = max(g, key=lambda e: (e[1], -e[3]))
        events.append(dict(pos=top[0], start=g[0][0], end=g[-1][0], type="close_encounter", other=top[2],
                           max_iou=round(top[1], 3),
                           detail=f"patient box close to track {top[2]} (max IoU {top[1]:.2f}, "
                                  f"centre distance {min(e[3] for e in g):.2f} box heights)"))
    return patient, source, events, alive, det_at

def finish_events(vid, a, raw, patient):
    """Merge nearby events, attach frames / times / priority and the patient track before & after."""
    fp, fps = a["frames_processed"], a["meta"]["fps"]
    for e in raw:
        e["priority"] = PRIORITY[e["type"]] + (e["max_iou"] if e["type"] == "close_encounter" else 0)
    raw = sorted(raw, key=lambda e: e["pos"])
    merged, gap = [], MERGE_S * fps
    for e in raw:
        if merged and fp[e["pos"]] - fp[merged[-1][-1]["pos"]] <= gap:
            merged[-1].append(e)
        else:
            merged.append([e])
    out = []
    for i, g in enumerate(merged):
        top = max(g, key=lambda e: e["priority"])
        s = min(e["start"] for e in g); en = max(e["end"] for e in g)
        before = next((int(patient[p]) for p in range(s - 1, -1, -1) if patient[p] >= 0), -1)
        after = next((int(patient[p]) for p in range(en + 1, len(patient)) if patient[p] >= 0), -1)
        types = list(dict.fromkeys(e["type"] for e in sorted(g, key=lambda e: -e["priority"])))
        ious = [e["max_iou"] for e in g if pd.notna(e["max_iou"])]
        out.append(dict(event_id=f"{vid}_e{i + 1:03d}", video_id=vid, frame_idx=int(fp[top["pos"]]),
                        t_s=round(fp[top["pos"]] / fps, 2), start_frame=int(fp[s]), end_frame=int(fp[en]),
                        types="+".join(types), priority=round(top["priority"] + 0.2 * (len(types) - 1), 2),
                        patient_track_before=before, patient_track_after=after,
                        other_track=int(top["other"]) if top["other"] is not None else -1,
                        max_iou=max(ious) if ious else np.nan, detail=" | ".join(dict.fromkeys(e["detail"] for e in g))))
    return out

t0 = time.time()
ALL_EVENTS, ALL_ANCHORS, ID_ROWS = [], [], []
if LABELS.empty:
    print("WARNING: no Stage 1 labels -> no anchors -> nobody can be marked as patient.")
for n_, vid in enumerate(VIDEO_IDS, 1):
    a = load(vid)
    T = track_video(a)
    anchors = anchors_for(a, T, vid)
    patient, source, raw, alive, det_at = assign_patient(a, T, anchors)
    events = finish_events(vid, a, raw, patient)
    fp, fps = a["frames_processed"], a["meta"]["fps"]
    T["role"] = (np.where(T["track_id"].to_numpy() == patient[T["pos"].to_numpy().astype(int)], "patient", "observer")
                 if len(T) else pd.Series(dtype=str))
    T.drop(columns="pos").to_csv(ID_DIR / "tracks" / f"{vid}.csv", index=False)
    pdet = np.array([det_at.get((int(t), p), -1) if t >= 0 else -1 for p, t in enumerate(patient)])
    pb = (np.where(pdet[:, None] >= 0, a["det_boxes"][np.maximum(pdet, 0)], np.nan).round(1)
          if len(a["det_boxes"]) else np.full((len(fp), 4), np.nan))
    pd.DataFrame({"frame_idx": fp, "t_s": (fp / fps).round(3), "patient_track": patient, "patient_det": pdet,
                  "x1": pb[:, 0], "y1": pb[:, 1], "x2": pb[:, 2], "y2": pb[:, 3], "source": source}
                 ).to_csv(ID_DIR / "patient" / f"{vid}.csv", index=False)
    fps_eff = fps / max(1, a["meta"].get("stride", 2))
    life = {t: int(v.sum()) for t, v in alive.items()}
    ID_ROWS.append(dict(video_id=vid, video_file=a["meta"].get("video", vid), fps=fps,
                        stride=a["meta"].get("stride", 2), width=a["meta"]["width"], height=a["meta"]["height"],
                        frames=len(fp), duration_s=round(len(fp) / fps_eff, 1), tracks=len(life),
                        tracks_ge_1s=sum(v >= fps_eff for v in life.values()),
                        patient_tracks=len(set(patient[patient >= 0].tolist())),
                        patient_coverage=round(float((patient >= 0).mean()), 3),
                        anchors_with_box=sum(an["has_box"] for an in anchors),
                        anchors_tracked=sum(an["kind"] == "tracked" for an in anchors),
                        anchor_conflicts=sum("anchor_conflict" in e["types"] for e in events),
                        anchor_mismatches=sum("anchor_mismatch" in e["types"] for e in events),
                        encounters=sum("close_encounter" in e["types"] for e in events),
                        events=len(events), suspects=sum(e["priority"] >= SUSPECT_MIN for e in events)))
    ALL_EVENTS += events
    ALL_ANCHORS += [{k: v for k, v in an.items() if k != "pos"} for an in anchors]
    if n_ % 10 == 0:
        print(f"  {n_}/{len(VIDEO_IDS)} videos")

IDSUM = pd.DataFrame(ID_ROWS).set_index("video_id")
EVENTS = pd.DataFrame(ALL_EVENTS, columns=["event_id", "video_id", "frame_idx", "t_s", "start_frame", "end_frame",
                                           "types", "priority", "patient_track_before", "patient_track_after",
                                           "other_track", "max_iou", "detail"])
ANCH = pd.DataFrame(ALL_ANCHORS)
IDSUM.to_csv(ID_DIR / "identity_summary.csv")
EVENTS.to_csv(ID_DIR / "events.csv", index=False)
ANCH.to_csv(ID_DIR / "anchors.csv", index=False)
(ID_DIR / "config.json").write_text(json.dumps(dict(
    tracker=TRACKER, TRACK_HIGH=TRACK_HIGH, TRACK_LOW=TRACK_LOW, TRACK_NEW=TRACK_NEW, TRACK_IOU=TRACK_IOU,
    MAX_AGE_S=MAX_AGE_S, ENC_IOU=ENC_IOU, ENC_DIST=ENC_DIST, JUMP_FRAC=JUMP_FRAC, MERGE_S=MERGE_S,
    IOU_MATCH=IOU_MATCH, scipy=linear_sum_assignment is not None, stage2_run=RUN_NAME,
    created=datetime.now().isoformat(timespec="seconds")), indent=2))

print(f"Done in {time.time() - t0:.0f} s with tracker '{TRACKER}'.")
print(f"Patient assigned in {IDSUM['patient_coverage'].mean():.1%} of processed frames (mean over videos); "
      f"{IDSUM['patient_tracks'].mean():.1f} patient track(s) per video.")
print(f"Suspected events: {len(EVENTS)} total, {int((EVENTS['priority'] >= SUSPECT_MIN).sum())} with priority >= "
      f"{SUSPECT_MIN}; anchor conflicts in {int((IDSUM['anchor_conflicts'] > 0).sum())} video(s).")
if len(ANCH):
    print("Anchors:", ANCH["kind"].value_counts().to_dict())
display(IDSUM.sort_values(["suspects", "anchor_conflicts"], ascending=False).head(15))
if len(EVENTS):
    display(EVENTS.sort_values("priority", ascending=False).head(15)[
        ["event_id", "t_s", "types", "priority", "patient_track_before", "patient_track_after", "other_track", "detail"]])
    d = IDSUM["suspects"].sort_values()
    d = d[d > 0]
    if len(d):
        fig, ax = plt.subplots(figsize=(9, max(2.5, 0.22 * len(d) + 1)))
        ax.barh(range(len(d)), d.values, color=C_BLUE, height=0.7)
        ax.set_yticks(range(len(d))); ax.set_yticklabels(d.index, fontsize=8); ax.grid(axis="y", visible=False)
        for i, v in enumerate(d.values):
            ax.text(v, i, f" {v}", va="center", fontsize=8, color=C_MUTED)
        ax.set_xlabel(f"suspected switch events (priority ≥ {SUSPECT_MIN})")
        ax.set_title("Where to look first with review_patient.py")
        savefig(fig, "identity_suspects_per_video")
print(f"\nNext: review the suspects ->  python review_patient.py   (reads {ID_DIR})")
_done('identity')

## 13. One video: who is who over time
Every track as a horizontal bar (**thick aqua = patient**, thin orange = observer), your labelled frames (magenta
dotted lines) and the suspected events (numbered red triangles). Below: frames at the top events with the patient
box in aqua and observers in orange (needs the video file).

In [ ]:
if "CELLS_DONE" not in globals():
    raise RuntimeError("Run cell 1 (Settings) first: the kernel was restarted or cells were skipped.")
_require('identity')

if len(IDSUM):
    vid = ID_VIEW_VIDEO if ID_VIEW_VIDEO in IDSUM.index else IDSUM["suspects"].idxmax()
    T = pd.read_csv(ID_DIR / "tracks" / f"{vid}.csv")
    fps = float(IDSUM.loc[vid, "fps"]); stride = int(IDSUM.loc[vid, "stride"])
    ev = EVENTS[EVENTS["video_id"] == vid].sort_values("frame_idx").reset_index(drop=True)
    an = ANCH[ANCH["video_id"] == vid] if len(ANCH) else pd.DataFrame(columns=["frame_idx"])
    print(f"=== {vid} ===  tracks {IDSUM.loc[vid, 'tracks']}, patient tracks {IDSUM.loc[vid, 'patient_tracks']}, "
          f"patient coverage {IDSUM.loc[vid, 'patient_coverage']:.0%}, events {len(ev)}")
    order = T.groupby("track_id")["frame_idx"].min().sort_values().index.tolist()
    fig, ax = plt.subplots(figsize=(16, max(2.5, 0.38 * len(order) + 1.4)))
    step = stride * 2.5                                     # bars break where a track is missing > 2 steps
    for y, tid in enumerate(order):
        g = T[T["track_id"] == tid].sort_values("frame_idx")
        f = g["frame_idx"].to_numpy(); role = g["role"].to_numpy()
        cut = np.where((np.diff(f) > step) | (role[1:] != role[:-1]))[0] + 1
        for seg_f, seg_r in zip(np.split(f, cut), np.split(role, cut)):
            pat = seg_r[0] == "patient"
            ax.hlines(y, seg_f[0] / fps, (seg_f[-1] + stride) / fps, color=C_PAT if pat else C_OBS,
                      lw=7 if pat else 3)
    for f in an["frame_idx"]:
        ax.axvline(f / fps, color="#e87ba4", ls=":", lw=1.2)
    for i, r in ev.iterrows():
        hot = r["priority"] >= SUSPECT_MIN
        ax.plot(r["frame_idx"] / fps, len(order) - 0.3, marker="v", ms=9 if hot else 6,
                color=C_BAD if hot else C_MUTED, zorder=5)
        ax.text(r["frame_idx"] / fps, len(order) + 0.15, str(i + 1), ha="center", fontsize=8,
                color=C_BAD if hot else C_MUTED)
    ax.set_yticks(range(len(order))); ax.set_yticklabels([f"track {t}" for t in order], fontsize=8)
    ax.set_ylim(-0.8, len(order) + 0.8); ax.set_xlabel("time (s)"); ax.grid(axis="y", visible=False)
    from matplotlib.lines import Line2D
    ax.legend(handles=[Line2D([], [], color=C_PAT, lw=7, label="patient"),
                       Line2D([], [], color=C_OBS, lw=3, label="observer"),
                       Line2D([], [], color="#e87ba4", ls=":", label="your labelled frame"),
                       Line2D([], [], color=C_BAD, marker="v", lw=0, label=f"suspect (priority ≥ {SUSPECT_MIN})")],
              loc="upper center", bbox_to_anchor=(0.5, -0.8 / fig.get_size_inches()[1]), ncol=4)
    ax.set_title(f"{vid}: who is the patient over time ({TRACKER})")
    savefig(fig, f"videos/{vid}_identity_timeline")
    if len(ev):
        show = ev.sort_values("priority", ascending=False).head(8).sort_values("frame_idx")
        display(show[["event_id", "t_s", "types", "priority", "patient_track_before", "patient_track_after", "detail"]])
        frames = read_frames(vid, show["frame_idx"].tolist())
        if frames:
            a = load(vid)
            cols = 4; nrow = (len(show) + cols - 1) // cols
            fig, axes = plt.subplots(nrow, cols, figsize=(18, 3.8 * nrow), squeeze=False)
            for axx in axes.ravel():
                axx.axis("off")
            for axx, (_, r) in zip(axes.ravel(), show.iterrows()):
                f = int(r["frame_idx"])
                if f not in frames:
                    continue
                img = frames[f].copy()
                s0, e0 = dets_at(a, f)
                tracked = set(T.loc[T["frame_idx"] == f, "det_idx"])
                for j in range(s0, e0):
                    if j not in tracked:
                        b = a["det_boxes"][j].astype(int); cv2.rectangle(img, (b[0], b[1]), (b[2], b[3]), BGR_OTHER, 1)
                for _, t in T[T["frame_idx"] == f].iterrows():
                    b = np.array([t.x1, t.y1, t.x2, t.y2]).astype(int); pat = t.role == "patient"
                    col = BGR_PAT if pat else BGR_OBS
                    cv2.rectangle(img, (b[0], b[1]), (b[2], b[3]), col, 4 if pat else 2)
                    cv2.putText(img, f"{'PATIENT' if pat else 'OBS'} #{int(t.track_id)}", (b[0], max(18, b[1] - 6)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, col, 2)
                axx.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
                axx.set_title(f"{r['event_id']}  t={r['t_s']}s  {r['types']}", fontsize=9,
                              color=C_BAD if r["priority"] >= SUSPECT_MIN else C_TEXT)
            plt.tight_layout()
            savefig(fig, f"videos/{vid}_identity_events")
        else:
            print("Original video not found in data/ -> event images skipped.")
_done('id-video')

## 14. Your review results (after `review_patient.py`)
Reads what you marked while watching (`identity/anomaly_review.csv`, `identity/video_review.csv`) and shows:
1. **Status per video**: finished OK / with anomalies / not finished, how much of each video you watched.
2. **Per anomaly type**: how many, in how many videos, total and typical duration, by layout.
3. **Automatic suspects vs your marks**: how many of your anomalies had an automatic suspect nearby
   (= could be found automatically) and which suspect types tend to be real.
4. **What makes each anomaly different** (input for a solution): patient box width / height / aspect ratio
   compared with that patient's usual box, overlap with other people, number of tracked people, ankle keypoint
   confidence — inside your marked periods vs normal frames of finished videos (`anomaly_features.csv`).
5. **Corrected patient tracks** in `identity/patient_corrected/` with a `quality` column per frame:
   `ok`, `corrected_label` (moved to the patient you clicked), `label_wrong` (label on the wrong person, real
   patient not clicked), `observer_in_box`, `patient_missing`, `other_anomaly`, or `not_reviewed`.
   Later stages can skip frames whose quality is not `ok` / `corrected_label`.

In [ ]:
if "CELLS_DONE" not in globals():
    raise RuntimeError("Run cell 1 (Settings) first: the kernel was restarted or cells were skipped.")
_require('identity')

ANOM_CSV, VREV_CSV = ID_DIR / "anomaly_review.csv", ID_DIR / "video_review.csv"
A_TYPES = ["label_pindah_ke_observer", "observer_mengisi_kotak_pasien", "pasien_hilang", "lainnya"]
QUALITY = {"label_pindah_ke_observer": "label_wrong", "observer_mengisi_kotak_pasien": "observer_in_box",
           "pasien_hilang": "patient_missing", "lainnya": "other_anomaly"}
FEATS = ["has_patient", "box_w_ratio", "box_h_ratio", "aspect_w_h", "max_iou_other", "n_people", "ankle_conf"]
HINTS = {  # (feature, direction) -> what it suggests
    ("box_w_ratio", +1): "patient box wider than usual -> box merged with / stretched over the observer; "
                         "try a width or aspect-ratio check, or split the box using keypoints",
    ("aspect_w_h", +1): "patient box unusually wide for its height -> likely two people in one box",
    ("box_h_ratio", -1): "patient box shorter than usual -> partly hidden (occlusion)",
    ("max_iou_other", +1): "patient box overlaps another person -> association by IoU is ambiguous; "
                           "try appearance (ReID) or keypoint-based matching",
    ("n_people", -1): "fewer people detected -> the observer was absorbed into the patient box; try a lower "
                      "NMS threshold or keep weak boxes",
    ("ankle_conf", -1): "ankle keypoints less confident -> feet hidden or keypoints taken from the other person",
    ("has_patient", -1): "patient often has no box -> detector misses or the track is lost",
}
def num_or(v, d=-1):
    try:
        return int(float(v))
    except (TypeError, ValueError):
        return d
def read_csv_or_empty(p, cols):
    return pd.read_csv(p, dtype={"video_id": str}) if p.exists() and p.stat().st_size > 1 else pd.DataFrame(columns=cols)

if not ANOM_CSV.exists() and not VREV_CSV.exists():
    print("No review yet. In a terminal, in the folder of this notebook, run:\n\n    python review_patient.py\n")
    print("Watch each video, mark anomalies with 1-4 (press again when it ends), press o when a video is done.\n"
          "Then run this cell again.")
else:
    AN = read_csv_or_empty(ANOM_CSV, ["anomaly_id", "video_id", "type", "start_frame", "end_frame", "duration_s",
                                      "patient_track_at_start", "true_patient_track", "observer_track"])
    VR = read_csv_or_empty(VREV_CSV, ["video_id", "status", "watched_pct"])
    for c in ("start_frame", "end_frame"):
        AN[c] = pd.to_numeric(AN[c], errors="coerce")
    AN["duration_s"] = pd.to_numeric(AN["duration_s"], errors="coerce")
    VR = VR.set_index("video_id")
    status = pd.Series({v: (VR.loc[v, "status"] if v in VR.index else "belum_selesai") for v in IDSUM.index})
    done_vids = status[status.isin(["ok", "ada_anomali"])].index.tolist()
    print(f"Videos: {len(status)} | finished {len(done_vids)} (ok {int((status == 'ok').sum())}, with anomalies "
          f"{int((status == 'ada_anomali').sum())}) | not finished {int((status == 'belum_selesai').sum())}")
    print(f"Anomalies marked: {len(AN)} in {AN['video_id'].nunique()} video(s)")

    # ---- 1. per video
    per_vid = pd.DataFrame({"status": status})
    per_vid["watched_pct"] = VR["watched_pct"].reindex(per_vid.index) if "watched_pct" in VR else np.nan
    for t in A_TYPES:
        sub = AN[AN["type"] == t]
        per_vid[t] = sub.groupby("video_id").size().reindex(per_vid.index).fillna(0).astype(int)
        per_vid[t + "_s"] = sub.groupby("video_id")["duration_s"].sum().reindex(per_vid.index).fillna(0).round(1)
    per_vid = per_vid.join(IDSUM[["duration_s", "suspects"]])
    if META is not None:
        per_vid = per_vid.join(META.set_index("video_id"), how="left")
    per_vid.to_csv(ID_DIR / "review_summary.csv")
    display(per_vid.sort_values(A_TYPES, ascending=False).head(20))

    # ---- 2. per type
    if len(AN):
        g = AN.groupby("type")
        by_type = pd.DataFrame({"marks": g.size(), "videos": g["video_id"].nunique(),
                                "total_s": g["duration_s"].sum().round(1),
                                "median_s": g["duration_s"].median().round(2)}).reindex(A_TYPES).dropna(how="all")
        print("\nPer anomaly type:")
        display(by_type)
        if META is not None and done_vids:
            grp = [c for c in META.columns if c != "video_id"][0]
            d = per_vid.loc[done_vids]
            print(f"\nMean anomalies per finished video, by {grp}:")
            display(d.groupby(grp)[A_TYPES].mean().round(2).assign(videos=d.groupby(grp).size()))

    # ---- 3. automatic suspects vs your marks (finished videos only)
    if len(AN) and len(EVENTS) and done_vids:
        def overlap(df, vid, s, e, fps):
            return df[(df["video_id"] == vid) & (df["end_frame"] >= s - fps) & (df["start_frame"] <= e + fps)]
        rows = []
        for _, r in AN[AN["video_id"].isin(done_vids)].iterrows():
            ev = overlap(EVENTS, r["video_id"], r["start_frame"], r["end_frame"], float(IDSUM.loc[r["video_id"], "fps"]))
            rows.append(dict(type=r["type"], any_suspect=len(ev) > 0,
                             strong_suspect=bool((ev["priority"] >= SUSPECT_MIN).any())))
        print("\nShare of your anomalies that had an automatic suspect within 1 s:")
        display(pd.DataFrame(rows).groupby("type")[["any_suspect", "strong_suspect"]].mean().round(3))
        ev_done = EVENTS[EVENTS["video_id"].isin(done_vids)].copy()
        ev_done["your_mark"] = [
            (lambda a_: a_["type"].iloc[0] if len(a_) else "no_anomaly")(
                overlap(AN, e.video_id, e.start_frame, e.end_frame, float(IDSUM.loc[e.video_id, "fps"])))
            for e in ev_done.itertuples(index=False)]
        ex = ev_done.assign(etype=ev_done["types"].str.split("+")).explode("etype").reset_index(drop=True)
        print("What you marked around each type of automatic suspect (share per row):")
        display(pd.crosstab(ex["etype"], ex["your_mark"], normalize="index").round(2))

    # ---- 4. features + 5. corrected tracks
    frames_all, anom_rows, n_fixed = [], [], 0
    for vid in IDSUM.index:
        P = pd.read_csv(ID_DIR / "patient" / f"{vid}.csv")
        fps = float(IDSUM.loc[vid, "fps"])
        an = AN[AN["video_id"] == vid].sort_values("start_frame")
        P["quality"] = "ok" if vid in done_vids else "not_reviewed"
        need_feats = vid in done_vids or len(an)
        if need_feats:
            T = pd.read_csv(ID_DIR / "tracks" / f"{vid}.csv")
            a = load(vid)
            w, h = P["x2"] - P["x1"], P["y2"] - P["y1"]
            F = pd.DataFrame({"video_id": vid, "frame_idx": P["frame_idx"], "has_patient": (P["patient_track"] >= 0).astype(float),
                              "box_w_ratio": w / np.nanmedian(w) if w.notna().any() else np.nan,
                              "box_h_ratio": h / np.nanmedian(h) if h.notna().any() else np.nan,
                              "aspect_w_h": w / h, "label": "normal"})
            byf = {f: g for f, g in T.groupby("frame_idx")}
            n_people, max_iou, ankle = [], [], []
            for f, pt, d, x1, y1, x2, y2 in P[["frame_idx", "patient_track", "patient_det", "x1", "y1", "x2", "y2"]].itertuples(index=False):
                g = byf.get(f)
                n_people.append(0 if g is None else len(g))
                if pt >= 0 and g is not None:
                    others = g.loc[g["track_id"] != pt, ["x1", "y1", "x2", "y2"]].to_numpy()
                    max_iou.append(float(iou_matrix([[x1, y1, x2, y2]], others).max()) if len(others) else 0.0)
                    ankle.append(float(a["det_kpt_scores"][int(d)][ANKLES].mean()) if d >= 0 else np.nan)
                else:
                    max_iou.append(np.nan); ankle.append(np.nan)
            F["n_people"], F["max_iou_other"], F["ankle_conf"] = n_people, max_iou, ankle
        if len(an):
            box_of = {(int(t), int(f)): (x1, y1, x2, y2, d) for t, f, x1, y1, x2, y2, d in
                      T[["track_id", "frame_idx", "x1", "y1", "x2", "y2", "det_idx"]].itertuples(index=False)}
            anchor_f = sorted(ANCH.loc[(ANCH["video_id"] == vid) & (ANCH["kind"] == "tracked"), "frame_idx"]) if len(ANCH) else []
            half = int(0.5 * fps)
            for _, r in an.iterrows():
                s, e = int(r["start_frame"]), int(r["end_frame"])
                if e - s < half:                                   # a single moment, not a period
                    if r["type"] == "label_pindah_ke_observer":     # wrong until the next labelled frame
                        e = next((f for f in anchor_f if f > s), int(P["frame_idx"].max()) + 1) - 1
                    else:
                        s, e = s - half, e + half
                win = (P["frame_idx"] >= s) & (P["frame_idx"] <= e)
                F.loc[win.to_numpy(), "label"] = r["type"]
                anom_rows.append(dict(anomaly_id=r["anomaly_id"], video_id=vid, type=r["type"],
                                      start_s=round(s / fps, 2), end_s=round(e / fps, 2), frames=int(win.sum()),
                                      observer_track=r.get("observer_track"), true_patient_track=r.get("true_patient_track"),
                                      **F.loc[win.to_numpy(), FEATS].mean().round(3).to_dict()))
                P.loc[win & (P["quality"] != "corrected_label"), "quality"] = QUALITY[r["type"]]
                if r["type"] == "label_pindah_ke_observer":
                    true, wrong = num_or(r.get("true_patient_track")), num_or(r.get("patient_track_at_start"))
                    target = win & ((P["patient_track"] == wrong) | (wrong < 0) | ((P["patient_track"] < 0) & (true >= 0)))
                    for i in P.index[target]:
                        f = int(P.at[i, "frame_idx"])
                        if true >= 0 and (true, f) in box_of:
                            x1, y1, x2, y2, d = box_of[(true, f)]
                            P.loc[i, ["patient_track", "patient_det", "x1", "y1", "x2", "y2"]] = [true, d, x1, y1, x2, y2]
                            P.at[i, "quality"] = "corrected_label"
                        else:
                            P.loc[i, ["patient_track", "patient_det", "x1", "y1", "x2", "y2"]] = [-1, -1, np.nan, np.nan, np.nan, np.nan]
                    n_fixed += int(target.sum())
                    if true >= 0:                                   # frames now on the real patient
                        P.loc[win & (P["patient_track"] == true), "quality"] = "corrected_label"
        P.to_csv(ID_DIR / "patient_corrected" / f"{vid}.csv", index=False)
        if vid in done_vids:
            frames_all.append(F)

    AF = pd.DataFrame(anom_rows)
    AF.to_csv(ID_DIR / "anomaly_features.csv", index=False)
    if frames_all:
        FF = pd.concat(frames_all, ignore_index=True)
        comp = FF.groupby("label")[FEATS].mean().round(3)
        comp.insert(0, "frames", FF.groupby("label").size())
        comp.to_csv(ID_DIR / "anomaly_feature_summary.csv")
        print("\nPatient box features: inside your marked anomalies vs normal frames (finished videos):")
        display(comp)
        normal = FF[FF["label"] == "normal"]
        if len(normal) and (comp.index != "normal").any():
            print("\nWhat stands out per anomaly type (biggest differences from normal frames):")
            for t in [x for x in A_TYPES if x in comp.index]:
                sub = FF[FF["label"] == t]
                eff = {}
                for c in FEATS:
                    if not (sub[c].notna().any() and normal[c].notna().any()):
                        continue
                    mn, mt = normal[c].mean(), sub[c].mean()
                    sd = max(float(np.nan_to_num(normal[c].std())), 0.05 * abs(mn), 1e-3)   # floor: avoid huge values
                    eff[c] = (mt - mn) / sd
                top = [kv for kv in sorted(eff.items(), key=lambda kv: -abs(kv[1])) if abs(kv[1]) >= 0.3][:3]
                print(f"\n  {t}  ({len(sub)} frames)")
                if not top:
                    print("    no feature differs clearly from normal frames (yet) -> mark more examples")
                for c, dlt in top:
                    sign = 1 if dlt > 0 else -1
                    mn, mt = normal[c].mean(), sub[c].mean()
                    rel = f"{100 * (mt - mn) / abs(mn):+.0f}%" if abs(mn) > 0.05 else f"{mt - mn:+.2f}"
                    print(f"    {c}: {mt:.2f} vs {mn:.2f} in normal frames ({rel}, effect {dlt:+.1f})"
                          + (f"\n      -> {HINTS[(c, sign)]}" if (c, sign) in HINTS else ""))
    print(f"\nCorrected {n_fixed} frames with a clicked real patient. Corrected tracks (+ quality per frame): "
          f"{ID_DIR / 'patient_corrected'}")
    print(f"Per anomaly features: {ID_DIR / 'anomaly_features.csv'}")
_done('review')